# Comparison with Observations: Survey Populations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings

## Load observed data

Load the ATNF catalog for the radio emission.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../examples/data/atnf_full_nobinary_13-09-2021.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
# Select only stars with measured P, Pdot, DM and radio flux.
# We also select only those that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["P0"]["[s]"].isin(['NAN'])]
df_atnf = df_atnf[~df_atnf["P1"]["[s/s]"].isin(['NAN'])]
df_atnf = df_atnf[~df_atnf["DM"]["[cm^-3pc]"].isin(['NAN'])]
df_atnf = df_atnf[~df_atnf["DIST"]["[kpc]"].isin(['NAN'])]
df_atnf = df_atnf[~df_atnf["S1400"]["[mJy]"].isin(['NAN'])]
df_atnf = df_atnf[~df_atnf["ASSOC"]["Unnamed: 24_level_1"].str.contains('|'.join(discard))]

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-19
df_atnf = df_atnf[df_atnf["P1"]["[s/s]"].to_numpy().astype(np.float) > 1.e-19]

In [ ]:
# Parks multibeam pulsar survey database.
df_atnf_pk = df_atnf[df_atnf['SURVEY']["Unnamed: 25_level_1"].str.contains("pksmb")]

# Database of pulsars detected by Parkes multibeam 
# that have a pulse width W10 measurement available.
df_atnf_pk_w10 = df_atnf_pk[~df_atnf_pk["W10"]["[ms]"].isin(['NAN'])]

# Database of pulsars detected by Parkes multibeam 
# that have a proper motion measurement available.
df_atnf_pk_pm = df_atnf_pk[~df_atnf_pk["PMRA"]["[mas/yr]"].isin(['NAN'])]
df_atnf_pk_pm = df_atnf_pk_pm[~df_atnf_pk["PMDEC"]["[mas/yr]"].isin(['NAN'])]

RA_pk_obs = df_atnf_pk["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_pk_obs = df_atnf_pk["DECJD"]["[deg]"].to_numpy().astype(np.float) 
l_pk_obs = df_atnf_pk["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_pk_obs = df_atnf_pk["Gb"]["[deg]"].to_numpy().astype(np.float) 
P_pk_obs = df_atnf_pk["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_pk_obs = df_atnf_pk["P1"]["[s/s]"].to_numpy().astype(np.float) 
DM_pk_obs = df_atnf_pk["DM"]["[cm^-3pc]"].to_numpy().astype(np.float)
dist_pk_obs = df_atnf_pk["DIST"]["[kpc]"].to_numpy().astype(np.float)
S1400_pk_obs = df_atnf_pk["S1400"]["[mJy]"].to_numpy().astype(np.float)

l_pk_obs_pm = df_atnf_pk_pm["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_pk_obs_pm = df_atnf_pk_pm["Gb"]["[deg]"].to_numpy().astype(np.float) 
Pdot_pk_obs_pm = df_atnf_pk_pm["P1"]["[s/s]"].to_numpy().astype(np.float) 
dist_pk_obs_pm = df_atnf_pk_pm["DIST"]["[kpc]"].to_numpy().astype(np.float)
pmRA_pk_obs = df_atnf_pk_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_pk_obs = df_atnf_pk_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float) 

l_pk_obs_w10 = df_atnf_pk_w10["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_pk_obs_w10 = df_atnf_pk_w10["Gb"]["[deg]"].to_numpy().astype(np.float) 
P_pk_obs_w10 = df_atnf_pk_w10["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_pk_obs_w10 = df_atnf_pk_w10["P1"]["[s/s]"].to_numpy().astype(np.float) 
dist_pk_obs_w10 = df_atnf_pk_w10["DIST"]["[kpc]"].to_numpy().astype(np.float)
w10_pk_obs = df_atnf_pk_w10["W10"]["[ms]"].to_numpy().astype(np.float)

# Convert galactic latitude in the range [-180., 180].
l_pk_obs[(l_pk_obs > 180.) & (l_pk_obs < 360.)] = (
    l_pk_obs[(l_pk_obs > 180.) & (l_pk_obs < 360.)] - 360.
)
l_pk_obs_pm[(l_pk_obs_pm > 180.) & (l_pk_obs_pm < 360.)] = (
    l_pk_obs_pm[(l_pk_obs_pm > 180.) & (l_pk_obs_pm < 360.)] - 360.
)
l_pk_obs_w10[(l_pk_obs_w10 > 180.) & (l_pk_obs_w10 < 360.)] = (
    l_pk_obs_w10[(l_pk_obs_w10 > 180.) & (l_pk_obs_w10 < 360.)] - 360.
)

# Select only pulsars falling in the Parkes multibeam sky coverage where completness is above 90%.
cond = (
    (l_pk_obs > -100.) & 
    (l_pk_obs < 50.) & 
    (np.abs(b_pk_obs) < 5.)
)
cond_w10 = (
    (l_pk_obs_w10 > -100.) & 
    (l_pk_obs_w10 < 50.) & 
    (np.abs(b_pk_obs_w10) < 5.)
)
cond_pm = (
    (l_pk_obs_pm > -100.) &
    (l_pk_obs_pm < 50.) & 
    (np.abs(b_pk_obs_pm) < 5.)
)

RA_pk_obs = RA_pk_obs[cond]
DEC_pk_obs = DEC_pk_obs[cond]
l_pk_obs = l_pk_obs[cond]
b_pk_obs = b_pk_obs[cond]
P_pk_obs = P_pk_obs[cond]
Pdot_pk_obs = Pdot_pk_obs[cond]
DM_pk_obs = DM_pk_obs[cond]
dist_pk_obs = dist_pk_obs[cond]
S1400_pk_obs = S1400_pk_obs[cond]
P_pk_obs_w10 = P_pk_obs_w10[cond_w10]
w10_pk_obs = w10_pk_obs[cond_w10]
pmRA_pk_obs = pmRA_pk_obs[cond_pm]
pmDEC_pk_obs = pmDEC_pk_obs[cond_pm]

number_pk = len(RA_pk_obs)

In [ ]:
# Swinburne multibeam pulsar survey database.
df_atnf_sw = df_atnf[df_atnf['SURVEY']["Unnamed: 25_level_1"].str.contains("pkssw")]
number_sw = len(df_atnf_sw)

# Database of pulsars detected by Swinburne that have a pulse width W10 measurment available.
df_atnf_sw_w10 = df_atnf_sw[~df_atnf_sw["W10"]["[ms]"].isin(['NAN'])]
# Database of pulsars detected by Swinburne that have a proper motion measurment available.
df_atnf_sw_pm = df_atnf_sw[~df_atnf_sw["PMRA"]["[mas/yr]"].isin(['NAN'])]
df_atnf_sw_pm = df_atnf_sw_pm[~df_atnf_sw["PMDEC"]["[mas/yr]"].isin(['NAN'])]

RA_sw_obs = df_atnf_sw["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_sw_obs = df_atnf_sw["DECJD"]["[deg]"].to_numpy().astype(np.float) 
l_sw_obs = df_atnf_sw["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_sw_obs = df_atnf_sw["Gb"]["[deg]"].to_numpy().astype(np.float) 
P_sw_obs = df_atnf_sw["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_sw_obs = df_atnf_sw["P1"]["[s/s]"].to_numpy().astype(np.float) 
DM_sw_obs = df_atnf_sw["DM"]["[cm^-3pc]"].to_numpy().astype(np.float)
dist_sw_obs = df_atnf_sw["DIST"]["[kpc]"].to_numpy().astype(np.float)
S1400_sw_obs = df_atnf_sw["S1400"]["[mJy]"].to_numpy().astype(np.float)

l_sw_obs_pm = df_atnf_sw_pm["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_sw_obs_pm = df_atnf_sw_pm["Gb"]["[deg]"].to_numpy().astype(np.float) 
Pdot_sw_obs_pm = df_atnf_sw_pm["P1"]["[s/s]"].to_numpy().astype(np.float) 
dist_sw_obs_pm = df_atnf_sw_pm["DIST"]["[kpc]"].to_numpy().astype(np.float)
pmRA_sw_obs = df_atnf_sw_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_sw_obs = df_atnf_sw_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float) 

l_sw_obs_w10 = df_atnf_sw_w10["Gl"]["[deg]"].to_numpy().astype(np.float) 
b_sw_obs_w10 = df_atnf_sw_w10["Gb"]["[deg]"].to_numpy().astype(np.float) 
P_sw_obs_w10 = df_atnf_sw_w10["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_sw_obs_w10 = df_atnf_sw_w10["P1"]["[s/s]"].to_numpy().astype(np.float) 
dist_sw_obs_w10 = df_atnf_sw_w10["DIST"]["[kpc]"].to_numpy().astype(np.float)
w10_sw_obs = df_atnf_sw_w10["W10"]["[ms]"].to_numpy().astype(np.float)

# Convert galactic latitude in the range [-180., 180].
l_sw_obs[(l_sw_obs > 180.) & (l_sw_obs < 360.)] = l_sw_obs[(l_sw_obs > 180.) & (l_sw_obs < 360.)] - 360.
l_sw_obs_pm[(l_sw_obs_pm > 180.) & (l_sw_obs_pm < 360.)] = l_sw_obs_pm[(l_sw_obs_pm > 180.) & (l_sw_obs_pm < 360.)] - 360.
l_sw_obs_w10[(l_sw_obs_w10 > 180.) & (l_sw_obs_w10 < 360.)] = l_sw_obs_w10[(l_sw_obs_w10 > 180.) & (l_sw_obs_w10 < 360.)] - 36

# Selection only pulsars falling in the Swinburne sky coverage where completness is above 90%.
cond = (l_sw_obs > -100.) & (l_sw_obs < 50.)
cond_w10 = (l_sw_obs_w10 > -100.) & (l_sw_obs_w10 < 50.)
cond_pm = (l_sw_obs_pm > -100.) & (l_sw_obs_pm < 50.)

RA_sw_obs = RA_sw_obs[cond]
DEC_sw_obs = DEC_sw_obs[cond]
l_sw_obs = l_sw_obs[cond]
b_sw_obs = b_sw_obs[cond]
P_sw_obs = P_sw_obs[cond]
Pdot_sw_obs = Pdot_sw_obs[cond]
DM_sw_obs = DM_sw_obs[cond]
dist_sw_obs = dist_sw_obs[cond]
S1400_sw_obs = S1400_sw_obs[cond]
P_sw_obs_w10 = P_sw_obs_w10[cond_w10]
w10_sw_obs = w10_sw_obs[cond_w10]
pmRA_sw_obs = pmRA_sw_obs[cond_pm]
pmDEC_sw_obs = pmDEC_sw_obs[cond_pm]

RA_all_obs = np.concatenate((RA_pk_obs, RA_sw_obs))
DEC_all_obs = np.concatenate((DEC_pk_obs, DEC_sw_obs))
pmRA_all_obs = np.concatenate((pmRA_pk_obs, pmRA_sw_obs))
pmDEC_all_obs = np.concatenate((pmDEC_pk_obs, pmDEC_sw_obs))
l_all_obs = np.concatenate((l_pk_obs, l_sw_obs))
b_all_obs = np.concatenate((b_pk_obs, b_sw_obs))
P_all_obs = np.concatenate((P_pk_obs, P_sw_obs))
Pdot_all_obs = np.concatenate((Pdot_pk_obs, Pdot_sw_obs))
DM_all_obs = np.concatenate((DM_pk_obs, DM_sw_obs))
dist_all_obs = np.concatenate((dist_pk_obs, dist_sw_obs))
S1400_all_obs = np.concatenate((S1400_pk_obs, S1400_sw_obs))
P_all_obs_w10 = np.concatenate((P_pk_obs_w10, P_sw_obs_w10))
w10_all_obs = np.concatenate((w10_pk_obs, w10_sw_obs))

number_sw = len(RA_sw_obs)

In [ ]:
print(f"Number of pulsars detected by Parks multibeam: {number_pk}")
print(f"Number of pulsars detected by Swinburne: {number_sw}")

## Load simulated data

Load the output of a simulated detection with the two surveys.

In [ ]:
# Select a `final_population.pkl.gz` file to import:
df_sim = pd.read_pickle(
    "../examples/data/simulation_full_example/final_population.pkl.gz",
    compression="gzip",
)

df_sim.head()

In [ ]:
# Extracting the parameters.
age = df_sim["age"]["[yr]"].to_numpy()
x = df_sim["x"]["[kpc]"].to_numpy()
y = df_sim["y"]["[kpc]"].to_numpy()
z = df_sim["z"]["[kpc]"].to_numpy()
RA = df_sim["RA"]["[deg]"].to_numpy()
DEC = df_sim["DEC"]["[deg]"].to_numpy()
l = df_sim["l"]["[deg]"].to_numpy()
b = df_sim["b"]["[deg]"].to_numpy()
pm_RA = df_sim["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = df_sim["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r = df_sim["v_r"]["[km s^-1]"].to_numpy()
v_phi = df_sim["v_phi"]["[km s^-1]"].to_numpy()
v_z = df_sim["v_z"]["[km s^-1]"].to_numpy()
DM = df_sim["DM"]["[pc cm^-3]"].to_numpy()
dist = df_sim["d"]["[kpc]"].to_numpy()
B = df_sim["B"]["[G]"].to_numpy()
chi = df_sim["chi"]["[rad]"].to_numpy()
P = df_sim["P"]["[s]"].to_numpy()
P_dot = df_sim["P_dot"]["[s yr^-1]"].to_numpy()
L_radio = df_sim["L_radio"]["[erg s^-1 Hz^-1]"].to_numpy()
S_radio_Jy = df_sim["S_radio"]["[Jy]"].to_numpy()
w_int = df_sim["w_int"]["[s]"].to_numpy()
detected_radio_pk = df_sim["detected_radio_PMPS"][" "].to_numpy(dtype=bool)
detected_radio_sw = df_sim["detected_radio_SMPS"][" "].to_numpy(dtype=bool)

detected_radio = detected_radio_pk | detected_radio_sw

In [ ]:
# Extracting parameters of detected objects.
RA_pk_sim = RA[detected_radio_pk]
DEC_pk_sim = DEC[detected_radio_pk]
pmRA_pk_sim = pm_RA[detected_radio_pk]
pmDEC_pk_sim = pm_DEC[detected_radio_pk]
l_pk_sim = l[detected_radio_pk]
b_pk_sim = b[detected_radio_pk]
P_pk_sim = P[detected_radio_pk]
Pdot_pk_sim = P_dot[detected_radio_pk]
DM_pk_sim = DM[detected_radio_pk]
dist_pk_sim = dist[detected_radio_pk]
S1400_pk_sim = S_radio_Jy[detected_radio_pk]
w_pk_sim = w_int[detected_radio_pk]

RA_sw_sim = RA[detected_radio_sw]
DEC_sw_sim = DEC[detected_radio_sw]
pmRA_sw_sim = pm_RA[detected_radio_sw]
pmDEC_sw_sim = pm_DEC[detected_radio_sw]
l_sw_sim = l[detected_radio_sw]
b_sw_sim = b[detected_radio_sw]
P_sw_sim = P[detected_radio_sw]
Pdot_sw_sim = P_dot[detected_radio_sw]
DM_sw_sim = DM[detected_radio_sw]
dist_sw_sim = dist[detected_radio_sw]
S1400_sw_sim = S_radio_Jy[detected_radio_sw]
w_sw_sim = w_int[detected_radio_sw]

RA_all_sim = np.concatenate((RA_pk_sim, RA_sw_sim))
DEC_all_sim = np.concatenate((DEC_pk_sim, DEC_sw_sim))
pmRA_all_sim = np.concatenate((pmRA_pk_sim, pmRA_sw_sim))
pmDEC_all_sim = np.concatenate((pmDEC_pk_sim, pmDEC_sw_sim))
l_all_sim = np.concatenate((l_pk_sim, l_sw_sim))
b_all_sim = np.concatenate((b_pk_sim, b_sw_sim))
P_all_sim = np.concatenate((P_pk_sim, P_sw_sim))
Pdot_all_sim = np.concatenate((Pdot_pk_sim, Pdot_sw_sim))
DM_all_sim = np.concatenate((DM_pk_sim, DM_sw_sim))
dist_all_sim = np.concatenate((dist_pk_sim, dist_sw_sim))
S1400_all_sim = np.concatenate((S1400_pk_sim, S1400_sw_sim))
w_all_sim = np.concatenate((w_pk_sim, w_sw_sim))

print(
    f"Number of pulsars detected by the simulated Parkes multibeam: {len(RA_pk_sim)}"
)
print(
    f"Number of pulsars detected by the simulated Swinburne: {len(RA_sw_sim)}"
)

Comparison of the sky distributions.

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_obs,
    DEC_pk_obs,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    RA_sw_obs,
    DEC_sw_obs,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_sim,
    DEC_pk_sim,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    RA_sw_sim,
    DEC_sw_sim,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_pk_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    RA_pk_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=False, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_sw_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    RA_sw_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=False, loc=2)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_pk_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    DEC_pk_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_sw_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    DEC_sw_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_obs,
    b_pk_obs,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l_sw_obs,
    b_sw_obs,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_sim,
    b_pk_sim,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    l_sw_sim,
    b_sw_sim,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
l_edges = np.linspace(-180.0, 180.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_pk_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    l_pk_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=False, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_sw_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    l_sw_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=False, loc=2)

plt.show()

In [ ]:
b_edges = np.linspace(-40.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_pk_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    b_pk_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=False, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_sw_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    b_sw_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=False, loc=2)

plt.show()

Compare proper motion distributions.

In [ ]:
pmRA_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_pk_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmRA_pk_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_sw_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmRA_sw_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
pmDEC_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_pk_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmDEC_pk_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_sw_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmDEC_sw_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=False, loc=0)

plt.show()

Compare DM distributions.

In [ ]:
dm_edges = np.linspace(0, 2500, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_pk_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    DM_pk_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_sw_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    DM_sw_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

Compare distance distributions.

In [ ]:
d_edges = np.linspace(0, 30, 36)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_pk_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    dist_pk_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=False, loc=0)

plt.show(block=False)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_sw_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    dist_sw_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=False, loc=0)

plt.show(block=False)

Comparing the spin-period and spin-period-derivative distributions.

In [ ]:
P_bins = np.logspace(-2.0, 2.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_pk_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    P_pk_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_sw_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    P_sw_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
Pdot_bins = np.logspace(-20.0, -8.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_pk_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Pdot_pk_sim / const.YR_TO_S,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_sw_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Pdot_sw_sim / const.YR_TO_S,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

Comparing PPdot diagrams.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_obs,
    Pdot_pk_obs,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1.0,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    P_sw_obs,
    Pdot_sw_obs,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed SMPS",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.set_ylim(1.0e-19, 1.0e-11)
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim,
    Pdot_pk_sim / const.YR_TO_S,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim / const.YR_TO_S,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=False, loc=0)

plt.show()

Comparing the spin-down power distributions.

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Computing the spin-down power.
Erot_dot_pk_sim = (
    NS_inertia
    * (2.0 * np.pi) ** 2
    * Pdot_pk_sim
    / const.YR_TO_S
    / (P_pk_sim ** 3)
)
Erot_dot_sw_sim = (
    NS_inertia
    * (2.0 * np.pi) ** 2
    * Pdot_sw_sim
    / const.YR_TO_S
    / (P_sw_sim ** 3)
)

Erot_dot_pk_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_pk_obs / (P_pk_obs ** 3)
)
Erot_dot_sw_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_sw_obs / (P_sw_obs ** 3)
)

In [ ]:
Erot_dot_bins = np.logspace(27.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_pk_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Erot_dot_pk_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_sw_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Erot_dot_sw_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

Comparing the radio-flux distributions.

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_pk_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    S1400_pk_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_sw_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    S1400_sw_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
L_pseudo_radio_pk_obs = S1400_pk_obs * const.MJY_TO_ERG * (dist_pk_obs * const.KPC_TO_CM)**2 * 1.374e9
L_pseudo_radio_sw_obs = S1400_sw_obs * const.MJY_TO_ERG * (dist_sw_obs * const.KPC_TO_CM)**2 * 1.374e9
L_pseudo_radio_pk_sim = S1400_pk_sim * 1000 * const.MJY_TO_ERG * (dist_pk_sim * const.KPC_TO_CM)**2 * 1.374e9
L_pseudo_radio_sw_sim = S1400_sw_sim * 1000 * const.MJY_TO_ERG * (dist_sw_sim * const.KPC_TO_CM)**2 * 1.374e9

eff_radio_pk_obs = L_pseudo_radio_pk_obs / Erot_dot_pk_obs
eff_radio_sw_obs = L_pseudo_radio_sw_obs / Erot_dot_sw_obs
eff_radio_pk_sim = L_pseudo_radio_pk_sim / Erot_dot_pk_sim
eff_radio_sw_sim = L_pseudo_radio_sw_sim / Erot_dot_sw_sim

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")
ax.loglog(
    Erot_dot_pk_obs, 
    eff_radio_pk_obs, 
    'o', 
    color='darkgray', 
    ms=6, 
    alpha=1, 
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    Erot_dot_pk_sim, 
    eff_radio_pk_sim, 
    'o', 
    color='tab:red', 
    ms=6, 
    alpha=1, 
    rasterized=True,
    label="Simulated PMPS",
)
ax.loglog(
    Erot_dot_sw_obs, 
    eff_radio_sw_obs, 
    'o', 
    color='black', 
    ms=6, 
    alpha=1, 
    rasterized=True,
    label="Observed SMPS",
)
ax.loglog(
    Erot_dot_sw_sim, 
    eff_radio_sw_sim, 
    'o', 
    color='tab:blue', 
    ms=6, 
    alpha=1, 
    rasterized=True,
    label="Simulated SMPS",
)
ax.axhline(
    y=1, 
    color='black', 
    linestyle='-', 
    linewidth=2
)
ax.axhline(
    y=0.1, 
    color='black', 
    linestyle='--', 
    linewidth=2
)

plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

Comparing the pulse-width distributions.

In [ ]:
# Converting pulse width into [deg].
w_pk_sim_deg = w_pk_sim / P_pk_sim * 360.0
w_sw_sim_deg = w_sw_sim / P_sw_sim * 360.0
w10_pk_obs_deg = w10_pk_obs / 1000 / P_pk_obs_w10 * 360.0
w10_sw_obs_deg = w10_sw_obs / 1000 / P_sw_obs_w10 * 360.0

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_obs_w10,
    w10_pk_obs_deg,
    "o",
    color="darkgray",
    ms=3,
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    P_sw_obs_w10,
    w10_sw_obs_deg,
    "x",
    color="black",
    ms=6,
    rasterized=True,
    label="Observed SMPS",
)
plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=False, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_sim,
    w_pk_sim_deg,
    "o",
    color="darkgray",
    ms=3,
    rasterized=True,
    label="Simulated PMPS",
)
ax.loglog(
    P_sw_sim,
    w_sw_sim_deg,
    "x",
    color="black",
    ms=6,
    rasterized=True,
    label="Simulated SMPS",
)
plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=False, loc=0)

plt.show()